# Solución Ejercicio 03b  |  Tema 4 - PRyES
## Estimación por máxima verosimilitud

stats

In [ ]:
# import required packages
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns
import pandas as pd
from scipy import optimize as opt  # para estimación por máxima verosimilitud

### Se sabe que la variable aleatoria X se distribuye como una triangular entre 10 y 20, pero se desconoce la moda. Se quiere estimar dicho parámetro de la distribución, para lo que se dispone de la muestra de datos almacenada en muestra_1000_triang_10_20.csv

In [ ]:
# Leemos los datos de la muestra
x = np.loadtxt('muestra_1000_triang_10_20.csv', delimiter=',', skiprows=1)

# tamaño de la muestra
n = len(x)
print(f'La muestra tiene {n} observaciones.')

### a) Representa el histograma de la muestra de datos disponible. ¿Se parece a una triangular?

In [ ]:
# Respuesta

nbins = 40 # número de intervalos, ajustar para buena visualización

fig = plt.figure(1, figsize=(14, 4)) # permite indicar el nº de la figura y las dimensiones (ancho y alto)
sns.histplot(data=x, bins=nbins)
sns.rugplot(x, color='r')  # opcional
plt.title(f'Histograma ({nbins} intervalos)')
plt.xlabel('x')
plt.ylabel('Frecuencia absoluta')
plt.show()

print('Sí parece que podría ser una distribución triangular.')

### b) Estima numéricamente por máxima verosimilitud, a partir de la muestra de 2000 observaciones, el valor de la moda de la distribución triangular que sigue X. Utiliza como punto inicial de búsqueda un valor razonable, deducido observando el histograma anterior.

In [ ]:
# Respuesta
# define la función de verosimilitud logarítmica para maximizarla

def verosimilitud_log(parameters):
    # sacamos los parámetros
    param1 = parameters
    
    # La función de verosimilitud logarítmica depende del problema y requiere
    # conocer la distribución de la v.a. bajo estudio, que dependerá de los params
    # a estimar por MV
    
    a = 10 # min
    c = param1 # moda 
    b = 20 # max

    loc = a
    escalado = b-a
    forma = (c-a)/escalado
    logLikelihood = np.sum(stats.triang.logpdf(x, loc=loc, scale=escalado, c = forma))  # <--- logpdf es el log de la pdf, ya está preparado ...
        
    # Como la función minimiza y queremos los params que maximizan la verosimilitud
    # simplemente la multimplicamos por -1: max(loglike) = min(-loglike)
    neg_logLikelihood = -1 * logLikelihood
    return neg_logLikelihood 

In [ ]:
# Maximiza la función de verosimilitud logarítmica (minimizando -1 * funcion)
# usamos como punto inicial de búsqueda
x0 = 13.5
mle_model = opt.minimize(verosimilitud_log, np.array(x0), method='L-BFGS-B')

# cogemos los coefs estimados
param1 = mle_model.x[0]

print(mle_model)
print('Parámetros estimados por MV:\n'
      f'  - parámetro 1: {param1}')

print(f'\nPor tanto, la moda estimada de la distribución triangular vale: {param1}')

### c) Representa en la misma figura el histograma de la muestra de la variable X y la función de densidad estimada, definida por la moda estimada por máxima verosilitud a partir de dicha muestra. Usa la opción stat="density" en la llamada a sns.histplot para mostrar frecuencias relativas en lugar de absolutas, ¿Tiene sentido el valor estimado?

In [ ]:
# Respuesta

# creamos la distribución
a = 10 # min
c = param1 # moda 
b = 20 # max

loc = a
escalado = b-a
forma = (c-a)/escalado
VA = stats.triang(loc=loc, scale=escalado, c = forma)

# representa ambas gráficas 
xs = np.linspace(VA.ppf(0.01)-1, VA.ppf(0.99)+1, 1000)

fig = plt.figure(1, figsize=(14, 4))
# mostramos densidad (i.e. freq. relativa)
nbins = 'auto'
sns.histplot(data=x, bins=nbins, stat="density", label = 'Histograma muestra')
sns.rugplot(x, color='r')  # opcional

# añade la distribución estimada
plt.plot( xs, VA.pdf(xs), color="black", linewidth = 4, label = f'Triang({a},{c},{b}) estimada')

plt.title(f'Histograma ({nbins} intervalos) y distribución estimada')
plt.xlabel('x')
plt.legend()
plt.ylabel('Frecuencia relativa')
plt.show()

print('Sí parece que está bien estimado. Si la muestra tuviese un mayor tamaño, habría menos dudas sobre la calidad del resultado.')

### d) Representa en la misma figura la función acumulada de probabilidad empírica (ECDF) de la variable X y la pdf estimada, definida por la moda estimada por máxima verosilitud a partir de dicha muestra. ¿Se parecen?¶

In [ ]:
# Respuesta
# para pintar la ECDF utiliza sns.ecdfplot(x, label = 'ECDF(x)')

# representa ambas gráficas 
xs = np.linspace(VA.ppf(0.01)-1, VA.ppf(0.99)+1, 1000)

fig = plt.figure(1, figsize=(14, 6))

# mostramos ecdf
sns.ecdfplot(x, label = 'ECDF(x)')
             
# añade la distribución estimada
plt.plot( xs, VA.cdf(xs), color="black", linewidth = 1, label = f'Triang({a},{c},{b}) estimada')

plt.title(f'ecdf y cdf de la distribución estimada')
plt.xlabel('x')
plt.legend()
plt.grid()
plt.ylabel('Probabilidad')
plt.show()

print('Aparentemente son muy parecidas.')